<a href="https://colab.research.google.com/github/gautamthampy/CMPE256-Group10/blob/collaborative_filtering_v1/recommender_submit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
data = {}
with open("/content/train-2.txt") as f:
  for line in f:
    parts = line.strip().split()
    if not parts:
      continue
    user = parts[0]
    items = parts[1:]
    data[user] = items

print(f"Number of users = {len(data.keys())}")

Number of users = 52643


In [2]:
from collections import Counter

full_pop_counts = Counter()
for items in data.values():
    full_pop_counts.update(items)

# sorted items by popularity (most common first)
all_items_by_pop_full = [it for it, _ in full_pop_counts.most_common()]

# for trimming users by popularity
full_pop_rank = {item: rank for rank, (item, _) in enumerate(full_pop_counts.most_common())}


In [3]:
MAX_ITEMS_PER_USER = 80
trimmed_full = {}
for user, items in data.items():
    items_sorted = sorted(items, key=lambda it: full_pop_rank[it])
    trimmed_full[user] = items_sorted[:MAX_ITEMS_PER_USER]


In [4]:
import itertools
from collections import defaultdict

cooc_full = defaultdict(lambda: defaultdict(int))

for items in trimmed_full.values():
    uniq_items = list(set(items))
    for i, j in itertools.combinations(uniq_items, 2):
        cooc_full[i][j] += 1
        cooc_full[j][i] += 1


In [5]:
MAX_NEIGHBORS = 400

for i, neighbors in list(cooc_full.items()):
    if len(neighbors) > MAX_NEIGHBORS:
        top_neighbors = sorted(neighbors.items(), key=lambda x: x[1], reverse=True)[:MAX_NEIGHBORS]
        cooc_full[i] = dict(top_neighbors)


In [6]:
from collections import defaultdict
import math

def recommend_popularity_full(user, k=20):
    seen = set(data[user])
    recs = []
    for it in all_items_by_pop_full:
        if it in seen:
            continue
        recs.append(it)
        if len(recs) == k:
            break
    return recs

def recommend_cooc_full(user, k=20):
    seen = set(data[user])
    scores = defaultdict(float)

    for i in seen:
        if i not in cooc_full:
            continue
        for j, c in cooc_full[i].items():
            if j in seen:
                continue
            scores[j] += c / math.sqrt(full_pop_counts[i] * full_pop_counts[j])

    if not scores:
        # fallback to popularity
        return recommend_popularity_full(user, k=k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    recs = [it for it, _ in ranked[:k]]

    # safety: if < k recs, pad with popularity
    if len(recs) < k:
        extra = recommend_popularity_full(user, k=k+50)
        for it in extra:
            if it not in seen and it not in recs:
                recs.append(it)
                if len(recs) == k:
                    break

    return recs


In [7]:
output_path = "/content/submission_score_calc_norm.txt"

with open(output_path, "w") as f:
    for user in data.keys():   # or a test_user_list if they give you one
        recs = recommend_cooc_full(user, k=20)
        line = user + " " + " ".join(recs) + "\n"
        f.write(line)

print("Wrote submission file to:", output_path)


Wrote submission file to: /content/submission_score_calc_norm.txt
